# 04 — Fixed Recirculation validation sweep and evaluation
Set the winning adapter in `configs/recirculation.yaml`. After the sweep, copy the selected layer pair and alpha into the config before final evaluation.

In [ ]:
REPO_URL = "https://github.com/seungjun-green/Korean-TDCS"
!git clone {REPO_URL} korean-math-tdcs
%cd korean-math-tdcs
!pip install -e .

In [ ]:
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

drive_results = Path("/content/drive/MyDrive/Korean-TDCS/results")
local_results = Path.cwd() / "results"
drive_results.mkdir(parents=True, exist_ok=True)

if local_results.is_symlink():
    if local_results.resolve() != drive_results.resolve():
        raise RuntimeError(f"{local_results} points to the wrong Drive directory")
elif local_results.exists():
    shutil.copytree(local_results, drive_results, dirs_exist_ok=True)
    shutil.rmtree(local_results)

if not local_results.exists():
    local_results.symlink_to(drive_results, target_is_directory=True)

print(f"Saving all outputs to {drive_results}")

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [ ]:
EVAL_BATCH_SIZE = 1  # Fixed Recirculation currently requires a batch size of 1.

sweep_cmd = ("python scripts/sweep_recirculation.py --config configs/recirculation.yaml "
             f"--set evaluation.batch_size={EVAL_BATCH_SIZE}")
!{sweep_cmd}

In [ ]:
import json

import pandas as pd

sweep = json.load(open('results/recirculation/sweep.json'))
pd.DataFrame(sweep['runs'])[['source_layer','destination_layer','alpha','score','tokens_per_second']].head(20)

In [ ]:
# After recording the best validation-only values in the YAML:
eval_cmd = ("python scripts/evaluate_recirculation.py "
            "--config configs/recirculation.yaml "
            f"--set evaluation.batch_size={EVAL_BATCH_SIZE}")
!{eval_cmd}